# Homework 2 — Prompt Engineering and RAG for Machine Translation

**WMT16 EN↔TR | Qwen 2.5 7B Instruct | MAPS + RAG**

He et al., TACL 2024 — *Exploring Human-Like Translation Strategy with LLMs*

---

## Hücre Çalıştırma Sırası

1. **Setup** (A, B, C) — her oturumda çalıştır
2. **Part 1** — veri yükle
3. **Part 2** — model yükle + benchmark
4. **Part 3** — zero-shot ve MAPS deneyleri
5. **Part 4** — RAG index + deney
6. **Part 5** — COMET skorlama + karşılaştırma

> **Not:** Her deney sonucu Drive'a JSONL olarak checkpoint'lenir.
> Oturum koparsa Part 5 verileri otomatik diskten yükler.

## A — Bağımlılıkları Kur

In [ ]:
# Temel paketler
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    datasets \
    sentence-transformers \
    "faiss-cpu>=1.8.0" \
    pyyaml numpy pandas tqdm scipy

# COMET: --no-deps ile bağımlılık çözücüyü bypass et,
# sonra sadece gerçekten eksik olanı kur
!pip install -q unbabel-comet --no-deps
!pip install -q "lightning>=2.0" torchmetrics sentencepiece

## B — Drive Mount ve Repo Clone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

REPO_DIR  = '/content/rag-mt-homework'
DRIVE_DIR = '/content/drive/MyDrive/RAG_Project'

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/mustafagalata/rag-mt-homework', REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Git clone başarısız (exit {result.returncode}):\n{result.stderr}')
    print('Repo klonlandı.')
else:
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        print(f'UYARI: git pull başarısız ({result.stderr}), mevcut dosyalarla devam ediliyor.')

if not os.path.exists(f'{REPO_DIR}/config.yaml'):
    raise FileNotFoundError(
        f'{REPO_DIR}/config.yaml bulunamadı — '
        'clone başarılı görünüyor ama repo boş olabilir. '
        'Yukarıdaki git çıktısını kontrol edin.'
    )

for d in ['data', 'indices', 'results', 'cache']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)

print(f'Repo hazır: {REPO_DIR}  |  Drive dizinleri: {DRIVE_DIR}')

## C — Import ve Config

In [ ]:
import sys, json, time, logging, os
from pathlib import Path
from tqdm.notebook import tqdm

# B hücresi çalışmadıysa (oturum yeniden başlatıldıysa) yeniden tanımla
REPO_DIR  = '/content/rag-mt-homework'
DRIVE_DIR = '/content/drive/MyDrive/RAG_Project'

if not Path(f'{REPO_DIR}/config.yaml').exists():
    raise EnvironmentError(
        f'{REPO_DIR}/config.yaml bulunamadı.\n'
        'Önce B hücresini çalıştırın (Drive mount → git clone).'
    )

sys.path.insert(0, REPO_DIR)

import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths'] = {
    'data_dir':    f'{DRIVE_DIR}/data',
    'index_dir':   f'{DRIVE_DIR}/indices',
    'results_dir': f'{DRIVE_DIR}/results',
    'cache_dir':   f'{DRIVE_DIR}/cache',
}
results_dir = cfg['paths']['results_dir']

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('Config yüklendi.')

---

## Part 1 — Dataset Selection and Preparation

In [ ]:
from src.data_loader import load_wmt16_splits, save_splits, load_splits_from_disk, print_sample_pairs

data_dir = cfg['paths']['data_dir']

if Path(f'{data_dir}/train.jsonl').exists():
    print('Diskten yükleniyor...')
    splits = load_splits_from_disk(data_dir)
else:
    print('WMT16 indiriliyor ve işleniyor...')
    splits = load_wmt16_splits(cfg, cache_dir=cfg['paths']['cache_dir'])
    save_splits(splits, data_dir)

print(f"Train: {len(splits['train']):,} | EN→TR test: {len(splits['test_en2tr']):,} | TR→EN test: {len(splits['test_tr2en']):,}")

In [ ]:
# Part 1.d — Örnek input-output çiftleri
print_sample_pairs(splits, n=5)

---

## Part 2 — Model Selection

In [ ]:
# GPU spesifikasyonları (Part 2.3.a-b için)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
from src.model import load_model_and_tokenizer, TranslationModel

model_raw, tokenizer = load_model_and_tokenizer(
    cfg['model']['name'],
    load_in_4bit=cfg['model']['load_in_4bit'],
    device_map=cfg['model']['device_map'],
    cache_dir=cfg['paths']['cache_dir'],
)
model = TranslationModel(model_raw, tokenizer, cfg)

In [ ]:
# ── Ön Benchmark: gerçek süreyi ölç, subset boyutunu doğrula ──
from src.prompts.zero_shot import build_zero_shot_prompt
from src.prompts.maps_strategy import MAPSPipeline

benchmark_pairs = splits['test_en2tr'][:cfg['benchmark']['n_sentences']]
maps_pipeline   = MAPSPipeline(model, cfg)

t0 = time.time()
for p in benchmark_pairs[:10]:
    model.generate(*build_zero_shot_prompt(p['en'], 'English', 'Turkish'))
zs_time = (time.time() - t0) / 10
print(f'Zero-shot: {zs_time:.1f} s/cümle')

t0 = time.time()
maps_pipeline.run(benchmark_pairs[0]['en'], 'English', 'Turkish')
maps_time = time.time() - t0
print(f'MAPS:      {maps_time:.1f} s/cümle')

n = cfg['data']['test_subset_size']
print(f'\nToplam tahmin ({n} cümle/yön × 2 yön):')
print(f'  Zero-shot : {zs_time  * n * 2 / 3600:.1f} saat')
print(f'  MAPS      : {maps_time * n * 2 / 3600:.1f} saat (bottleneck)')

---

## Part 3 — Prompt Engineering

### Part 3.D.1 — Zero-Shot Experiment

In [ ]:
from src.prompts.zero_shot import build_zero_shot_prompt

def run_zero_shot_with_checkpoint(pairs, direction, model, results_dir, checkpoint_every=50):
    src_lang = 'English' if direction == 'en2tr' else 'Turkish'
    tgt_lang = 'Turkish' if direction == 'en2tr' else 'English'
    src_key  = 'en'      if direction == 'en2tr' else 'tr'
    ref_key  = 'tr'      if direction == 'en2tr' else 'en'

    out_path = Path(f'{results_dir}/zero_shot_{direction}.jsonl')

    done_indices = set()
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                try:
                    done_indices.add(json.loads(line)['idx'])
                except (json.JSONDecodeError, KeyError):
                    pass
        print(f'[{direction}] Resume: {len(done_indices)}/{len(pairs)} cümle zaten mevcut.')

    pending = [(i, p) for i, p in enumerate(pairs) if i not in done_indices]
    print(f'[{direction}] İşlenecek: {len(pending)} cümle')

    if pending:
        with open(out_path, 'a', encoding='utf-8') as f:
            for count, (i, p) in enumerate(tqdm(pending, desc=f'Zero-shot {direction}'), 1):
                hyp = model.generate(*build_zero_shot_prompt(p[src_key], src_lang, tgt_lang))
                record = {'idx': i, 'src': p[src_key], 'ref': p[ref_key], 'hyp': hyp}
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                if count % checkpoint_every == 0:
                    f.flush()
                    print(f'  ✓ Checkpoint: {len(done_indices) + count}/{len(pairs)} kaydedildi')

    results = []
    with open(out_path, encoding='utf-8') as f:
        for line in f:
            results.append(json.loads(line))
    results.sort(key=lambda x: x['idx'])
    return results

zs_en2tr = run_zero_shot_with_checkpoint(splits['test_en2tr'], 'en2tr', model, results_dir)
zs_tr2en = run_zero_shot_with_checkpoint(splits['test_tr2en'], 'tr2en', model, results_dir)
print('Zero-shot tamamlandı.')

### Part 3.D.2 — MAPS Experiment

In [ ]:
from src.prompts.maps_strategy import MAPSPipeline

maps_pipeline = MAPSPipeline(model, cfg)

def run_maps_with_checkpoint(pairs, direction, pipeline, results_dir, checkpoint_every=50):
    src_lang = 'English' if direction == 'en2tr' else 'Turkish'
    tgt_lang = 'Turkish' if direction == 'en2tr' else 'English'
    src_key  = 'en'      if direction == 'en2tr' else 'tr'
    ref_key  = 'tr'      if direction == 'en2tr' else 'en'

    out_path = Path(f'{results_dir}/maps_{direction}.jsonl')

    # Kaldığı yerden devam: hangi index'ler işlenmiş?
    done_indices = set()
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                try:
                    done_indices.add(json.loads(line)['idx'])
                except (json.JSONDecodeError, KeyError):
                    pass
        print(f'[{direction}] Resume: {len(done_indices)}/{len(pairs)} cümle zaten mevcut.')

    pending = [(i, p) for i, p in enumerate(pairs) if i not in done_indices]
    print(f'[{direction}] İşlenecek: {len(pending)} cümle')

    if pending:
        with open(out_path, 'a', encoding='utf-8') as f:
            for count, (i, p) in enumerate(tqdm(pending, desc=f'MAPS {direction}'), 1):
                r = pipeline.run(p[src_key], src_lang, tgt_lang)
                record = {
                    'idx': i,
                    'src': r.src,
                    'ref': p[ref_key],
                    'hyp': r.final_translation,
                    'selected_index': r.selected_index,
                    'candidates': r.candidates,
                    'keywords': r.keywords,
                    'topic': r.topic,
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

                if count % checkpoint_every == 0:
                    f.flush()   # Drive'a zorla yaz
                    total_done = len(done_indices) + count
                    print(f'  ✓ Checkpoint: {total_done}/{len(pairs)} kaydedildi')

    # Tüm sonuçları sıralı oku ve döndür
    results = []
    with open(out_path, encoding='utf-8') as f:
        for line in f:
            results.append(json.loads(line))
    results.sort(key=lambda x: x['idx'])
    return results

maps_en2tr = run_maps_with_checkpoint(
    splits['test_en2tr'], 'en2tr', maps_pipeline, results_dir
)
maps_tr2en = run_maps_with_checkpoint(
    splits['test_tr2en'], 'tr2en', maps_pipeline, results_dir
)
print('MAPS tamamlandı.')

---

## Part 4 — RAG

### Part 4.B — RAG Index Oluşturma

In [ ]:
from src.rag.embedder import Embedder
from src.rag.indexer import build_index, save_index, load_index

index_dir = cfg['paths']['index_dir']
embedder  = Embedder(cfg['rag']['embedding_model'], cache_dir=cfg['paths']['cache_dir'])

# EN→TR index: kaynak dil İngilizce → İngilizce cümleler embed edilir
if not Path(f'{index_dir}/en2tr.faiss').exists():
    print('EN→TR index oluşturuluyor...')
    en_embs  = embedder.embed_passages([p['en'] for p in splits['train']],
                                        batch_size=cfg['rag']['batch_size_embed'])
    index_en = build_index(en_embs)
    save_index(index_en, splits['train'], index_dir, 'en2tr')
    train_pairs_en = splits['train']          # bellekte sakla
else:
    print('EN→TR index diskten yükleniyor...')
    index_en, train_pairs_en = load_index(index_dir, 'en2tr')

# TR→EN index: kaynak dil Türkçe → Türkçe cümleler embed edilir
if not Path(f'{index_dir}/tr2en.faiss').exists():
    print('TR→EN index oluşturuluyor...')
    tr_embs  = embedder.embed_passages([p['tr'] for p in splits['train']],
                                        batch_size=cfg['rag']['batch_size_embed'])
    index_tr = build_index(tr_embs)
    save_index(index_tr, splits['train'], index_dir, 'tr2en')
    train_pairs_tr = splits['train']          # bellekte sakla
else:
    print('TR→EN index diskten yükleniyor...')
    index_tr, train_pairs_tr = load_index(index_dir, 'tr2en')

print('Index hazır.')

### Part 4.B — RAG Experiment

In [ ]:
from src.rag.retriever import Retriever
from src.prompts.rag_few_shot import build_rag_prompt

retriever_en = Retriever(index_en, train_pairs_en, top_k=cfg['rag']['top_k'])
retriever_tr = Retriever(index_tr, train_pairs_tr, top_k=cfg['rag']['top_k'])

def run_rag_with_checkpoint(pairs, direction, retriever, embedder, model, results_dir, checkpoint_every=50):
    src_lang = 'English' if direction == 'en2tr' else 'Turkish'
    tgt_lang = 'Turkish' if direction == 'en2tr' else 'English'
    src_key  = 'en'      if direction == 'en2tr' else 'tr'
    ref_key  = 'tr'      if direction == 'en2tr' else 'en'

    out_path = Path(f'{results_dir}/rag_{direction}.jsonl')

    done_indices = set()
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                try:
                    done_indices.add(json.loads(line)['idx'])
                except (json.JSONDecodeError, KeyError):
                    pass
        print(f'[{direction}] Resume: {len(done_indices)}/{len(pairs)} cümle zaten mevcut.')

    pending = [(i, p) for i, p in enumerate(pairs) if i not in done_indices]
    print(f'[{direction}] İşlenecek: {len(pending)} cümle')

    if pending:
        with open(out_path, 'a', encoding='utf-8') as f:
            for count, (i, p) in enumerate(tqdm(pending, desc=f'RAG {direction}'), 1):
                src       = p[src_key]
                q_emb     = embedder.embed_queries([src])
                retrieved = retriever.retrieve(q_emb)
                hyp       = model.generate(*build_rag_prompt(src, src_lang, tgt_lang, retrieved))
                record = {
                    'idx': i, 'src': src, 'ref': p[ref_key], 'hyp': hyp,
                    'retrieved_scores': [r['score'] for r in retrieved],
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                if count % checkpoint_every == 0:
                    f.flush()
                    print(f'  ✓ Checkpoint: {len(done_indices) + count}/{len(pairs)} kaydedildi')

    results = []
    with open(out_path, encoding='utf-8') as f:
        for line in f:
            results.append(json.loads(line))
    results.sort(key=lambda x: x['idx'])
    return results

rag_en2tr = run_rag_with_checkpoint(splits['test_en2tr'], 'en2tr', retriever_en, embedder, model, results_dir)
rag_tr2en = run_rag_with_checkpoint(splits['test_tr2en'], 'tr2en', retriever_tr, embedder, model, results_dir)
print('RAG tamamlandı.')

---

## Part 5 — Experimental Comparison

In [ ]:
# ── Oturum kopması durumunda diskten yükle ──
def _load_jsonl(name):
    path = Path(f'{results_dir}/{name}.jsonl')
    if not path.exists():
        raise FileNotFoundError(f'{name}.jsonl bulunamadı. Önce ilgili deneyi çalıştırın.')
    with open(path, encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    records.sort(key=lambda x: x['idx'])
    return records

try:
    zs_en2tr
except NameError:
    print('Sonuçlar bellekte yok, diskten yükleniyor...')
    zs_en2tr   = _load_jsonl('zero_shot_en2tr')
    zs_tr2en   = _load_jsonl('zero_shot_tr2en')
    maps_en2tr = _load_jsonl('maps_en2tr')
    maps_tr2en = _load_jsonl('maps_tr2en')
    rag_en2tr  = _load_jsonl('rag_en2tr')
    rag_tr2en  = _load_jsonl('rag_tr2en')
    print('Yüklendi.')
else:
    print('Sonuçlar zaten bellekte.')

In [ ]:
# ── GPU belleğini temizle, COMET'e yer aç ──
import torch, gc
try:
    del model_raw
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print('GPU belleği temizlendi.')

In [ ]:
from src.evaluation.comet_scorer import load_comet_model, score_direction

comet_model = load_comet_model(
    cfg['evaluation']['comet_model'],
    cache_dir=cfg['paths']['cache_dir'],
)

In [ ]:
import pandas as pd

comet_results = {}
batch = cfg['evaluation']['comet_batch_size']

experiments = [
    ('zero_shot', 'en2tr', zs_en2tr,   splits['test_en2tr']),
    ('zero_shot', 'tr2en', zs_tr2en,   splits['test_tr2en']),
    ('maps',      'en2tr', maps_en2tr, splits['test_en2tr']),
    ('maps',      'tr2en', maps_tr2en, splits['test_tr2en']),
    ('rag',       'en2tr', rag_en2tr,  splits['test_en2tr']),
    ('rag',       'tr2en', rag_tr2en,  splits['test_tr2en']),
]

for method, direction, outputs, pairs in experiments:
    hyps   = [r['hyp'] for r in outputs]
    result = score_direction(comet_model, pairs, hyps, direction, batch_size=batch)
    key    = f'{method}_{direction}'
    comet_results[key] = result['system_score']
    print(f'{key:30s}: COMET = {result["system_score"]:.4f}')

with open(f'{results_dir}/comet_scores.json', 'w') as f:
    json.dump(comet_results, f, indent=2)

rows = [
    {
        'Direction':  'EN→TR' if d == 'en2tr' else 'TR→EN',
        'Zero-Shot':  round(comet_results[f'zero_shot_{d}'], 4),
        'MAPS':       round(comet_results[f'maps_{d}'],      4),
        'RAG 5-shot': round(comet_results[f'rag_{d}'],       4),
    }
    for d in ['en2tr', 'tr2en']
]
print('\n', pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Örnek çeviri karşılaştırması (rapora kopyalanacak)
for idx in range(3):
    p = splits['test_en2tr'][idx]
    print(f'[{idx}] Source    : {p["en"]}')
    print(f'    Reference : {p["tr"]}')
    print(f'    Zero-shot : {zs_en2tr[idx]["hyp"]}')
    print(f'    MAPS      : {maps_en2tr[idx]["hyp"]}')
    print(f'    RAG 5-shot: {rag_en2tr[idx]["hyp"]}')
    print()

---

## Part 6 — Report Summary

In [ ]:
import datetime

lines = []
lines.append(f"# Homework 2 — Results Summary")
lines.append(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n")

# ── COMET Tablosu ──────────────────────────────────────────────────────────
lines.append("## COMET Scores (wmt22-comet-da)\n")
lines.append(f"| Direction | Zero-Shot | MAPS  | RAG 5-shot |")
lines.append(f"|-----------|-----------|-------|------------|")
for d, label in [('en2tr', 'EN → TR'), ('tr2en', 'TR → EN')]:
    zs  = comet_results.get(f'zero_shot_{d}', float('nan'))
    mp  = comet_results.get(f'maps_{d}',      float('nan'))
    rg  = comet_results.get(f'rag_{d}',       float('nan'))
    lines.append(f"| {label}    | {zs:.4f}    | {mp:.4f} | {rg:.4f}     |")

lines.append("")

# ── Örnek Çeviriler ────────────────────────────────────────────────────────
lines.append("## Example Translations (EN → TR, first 5 sentences)\n")
for idx in range(5):
    p = splits['test_en2tr'][idx]
    lines.append(f"### [{idx+1}]")
    lines.append(f"**Source:**    {p['en']}")
    lines.append(f"**Reference:** {p['tr']}")
    lines.append(f"**Zero-shot:** {zs_en2tr[idx]['hyp']}")
    lines.append(f"**MAPS:**      {maps_en2tr[idx]['hyp']}")
    lines.append(f"**RAG:**       {rag_en2tr[idx]['hyp']}")
    lines.append("")

lines.append("## Example Translations (TR → EN, first 5 sentences)\n")
for idx in range(5):
    p = splits['test_tr2en'][idx]
    lines.append(f"### [{idx+1}]")
    lines.append(f"**Source:**    {p['tr']}")
    lines.append(f"**Reference:** {p['en']}")
    lines.append(f"**Zero-shot:** {zs_tr2en[idx]['hyp']}")
    lines.append(f"**MAPS:**      {maps_tr2en[idx]['hyp']}")
    lines.append(f"**RAG:**       {rag_tr2en[idx]['hyp']}")
    lines.append("")

# ── Dataset İstatistikleri ─────────────────────────────────────────────────
lines.append("## Dataset Statistics\n")
lines.append(f"- RAG corpus (train): {len(splits['train']):,} sentence pairs")
lines.append(f"- Test EN→TR:         {len(splits['test_en2tr']):,} sentence pairs")
lines.append(f"- Test TR→EN:         {len(splits['test_tr2en']):,} sentence pairs")
lines.append(f"- RAG top-k:          {cfg['rag']['top_k']}")
lines.append(f"- Embedding model:    {cfg['rag']['embedding_model']}")
lines.append(f"- LLM:                {cfg['model']['name']} (4-bit NF4)")

report_path = f"{results_dir}/report_summary.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f"Rapor kaydedildi: {report_path}")